# Тема 8. StudyMate — освітній асистент — домашня робота
**Курс:** AI Fundamentals · Product Management

⚡ Запускай комірки по порядку зверху вниз (Shift+Enter)

## 0. Налаштування

In [1]:
!pip install -q \
    "langchain==1.3.11" \
    "langchain-openai==1.3.3" \
    "langchain-community==0.4.2" \
    "langgraph==1.2.7" \
    "langchainhub==0.1.21" \
    "python-dotenv==1.2.2" \
    "wikipedia==1.4.0" \
    "numexpr==2.14.1" \
    "numpy<3" 2>&1 | grep -v "dependency conflicts"

In [2]:
import os
from datetime import datetime
from typing import List, Dict, Any
import re
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

print("✅ Імпорти виконано.")

✅ Імпорти виконано.


In [3]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ Ключ з Colab Secrets")
except Exception:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
    print("✅ Ключ встановлено")

✅ Ключ з Colab Secrets


## 1. Бази даних

**Примітка щодо хімії.** У продуктовому баченні StudyMate (HW2-HW3) хімія свідомо виключена зі скоупу через обмежену технічну спроможність автора перевіряти хімічний контент (немає технічної освіти в цій галузі). У цьому ДЗ хімія додана виключно тому, що методичка Варіанту 3 вимагає інструмент `periodic_table` і формули з хімії як частину технічного завдання курсу. Для реального продукту цей розділ бази даних не використовувався б.

In [4]:
FORMULAS_DB = {
    "математика": {
        "площа кола": {
            "формула": "S = πr²",
            "змінні": {"S": "площа", "π": "число пі (≈3.14159)", "r": "радіус"},
            "приклад": "Якщо r = 5 см, то S = π × 5² = 78.54 см²",
        },
        "теорема піфагора": {
            "формула": "a² + b² = c²",
            "змінні": {"a, b": "катети", "c": "гіпотенуза"},
            "приклад": "Якщо a = 3, b = 4, то c = √(9 + 16) = 5",
            "примітка": "Працює лише для прямокутних трикутників",
        },
        "квадратне рівняння": {
            "формула": "x = (-b ± √(b² - 4ac)) / 2a",
            "змінні": {"a, b, c": "коефіцієнти рівняння ax² + bx + c = 0"},
            "приклад": "Для x² - 5x + 6 = 0: x = (5 ± √1)/2, x₁ = 3, x₂ = 2",
            "примітка": "Дискримінант D = b² - 4ac визначає кількість коренів",
        },
        "об'єм кулі": {
            "формула": "V = (4/3)πr³",
            "змінні": {"V": "об'єм", "r": "радіус"},
            "приклад": "Якщо r = 3 см, то V = (4/3) × π × 27 ≈ 113.1 см³",
        },
        "сума арифметичної прогресії": {
            "формула": "Sₙ = n(a₁ + aₙ) / 2",
            "змінні": {"Sₙ": "сума перших n членів", "a₁": "перший член", "aₙ": "n-й член", "n": "кількість членів"},
            "приклад": "a₁=2, aₙ=20, n=10: S = 10 × 22 / 2 = 110",
        },
        "сума геометричної прогресії": {
            "формула": "Sₙ = a₁(qⁿ - 1) / (q - 1)",
            "змінні": {"Sₙ": "сума перших n членів", "a₁": "перший член", "q": "знаменник прогресії", "n": "кількість членів"},
            "приклад": "a₁=1, q=2, n=5: S = 1×(32-1)/(2-1) = 31",
            "примітка": "При q = 1 формула не застосовується (діли на нуль), сума дорівнює n × a₁",
        },
        "скалярний добуток векторів": {
            "формула": "a·b = |a||b|cos(φ)",
            "змінні": {"a·b": "скалярний добуток", "|a|, |b|": "довжини векторів", "φ": "кут між векторами"},
            "приклад": "|a|=3, |b|=4, φ=60°: a·b = 3×4×0.5 = 6",
            "примітка": "Якщо a·b = 0, вектори перпендикулярні",
        },
        "визначник матриці 2×2": {
            "формула": "det(A) = ad - bc",
            "змінні": {"a, b, c, d": "елементи матриці [[a,b],[c,d]]"},
            "приклад": "Для [[2,3],[1,4]]: det = 2×4 - 3×1 = 5",
            "примітка": "Якщо det(A) = 0, матриця вироджена (не має оберненої)",
        },
        "похідна степеневої функції": {
            "формула": "(xⁿ)' = n·xⁿ⁻¹",
            "змінні": {"x": "змінна", "n": "показник степеня"},
            "приклад": "(x³)' = 3x²",
            "примітка": "Базове правило диференціювання, основа для складніших похідних",
        },
        "границя числової послідовності": {
            "формула": "lim(n→∞) 1/n = 0",
            "змінні": {"n": "номер члена послідовності"},
            "приклад": "Послідовність 1, 1/2, 1/3, ... прямує до 0 при зростанні n",
            "примітка": "Класичний приклад збіжної послідовності",
        },
    },
    "фізика": {
        "кінетична енергія": {
            "формула": "Eₖ = mv² / 2",
            "змінні": {"Eₖ": "кінетична енергія (Дж)", "m": "маса (кг)", "v": "швидкість (м/с)"},
            "приклад": "Тіло 2 кг зі швидкістю 3 м/с: Eₖ = 2 × 9 / 2 = 9 Дж",
        },
        "потенціальна енергія": {
            "формула": "Eₚ = mgh",
            "змінні": {"Eₚ": "потенціальна енергія (Дж)", "m": "маса (кг)", "g": "прискорення вільного падіння (≈9.8 м/с²)", "h": "висота (м)"},
            "приклад": "Тіло 5 кг на висоті 10 м: Eₚ = 5 × 9.8 × 10 = 490 Дж",
        },
        "закон ома": {
            "формула": "I = U / R",
            "змінні": {"I": "сила струму (А)", "U": "напруга (В)", "R": "опір (Ом)"},
            "приклад": "При напрузі 12 В та опорі 4 Ом: I = 12 / 4 = 3 А",
        },
        "швидкість": {
            "формула": "v = s / t",
            "змінні": {"v": "швидкість (м/с)", "s": "шлях (м)", "t": "час (с)"},
            "приклад": "Якщо s = 100 м, t = 10 с, то v = 10 м/с",
        },
        "другий закон ньютона": {
            "формула": "F = ma",
            "змінні": {"F": "сила (Н)", "m": "маса (кг)", "a": "прискорення (м/с²)"},
            "приклад": "Тіло 10 кг з прискоренням 2 м/с²: F = 20 Н",
        },
        "закон всесвітнього тяжіння": {
            "формула": "F = G·m₁m₂ / r²",
            "змінні": {"F": "сила тяжіння (Н)", "G": "гравітаційна стала (≈6.674×10⁻¹¹)", "m₁, m₂": "маси тіл (кг)", "r": "відстань між тілами (м)"},
            "приклад": "Використовується для розрахунку орбіт планет та ваги тіл",
        },
        "рівняння стану ідеального газу": {
            "формула": "pV = nRT",
            "змінні": {"p": "тиск (Па)", "V": "об'єм (м³)", "n": "кількість речовини (моль)", "R": "універсальна газова стала (≈8.314 Дж/(моль·К))", "T": "температура (К)"},
            "приклад": "Рівняння Менделєєва-Клапейрона, зв'язує тиск, об'єм і температуру газу",
            "примітка": "Працює для ідеального газу, реальні гази відхиляються при високому тиску",
        },
        "закон кулона": {
            "формула": "F = k·q₁q₂ / r²",
            "змінні": {"F": "сила взаємодії зарядів (Н)", "k": "коефіцієнт (≈9×10⁹ Н·м²/Кл²)", "q₁, q₂": "величини зарядів (Кл)", "r": "відстань між зарядами (м)"},
            "приклад": "Описує взаємодію двох точкових електричних зарядів",
        },
        "довжина хвилі": {
            "формула": "λ = v / f",
            "змінні": {"λ": "довжина хвилі (м)", "v": "швидкість поширення хвилі (м/с)", "f": "частота (Гц)"},
            "приклад": "Звук у повітрі (v≈340 м/с) з частотою 340 Гц: λ = 1 м",
        },
        "релятивістська енергія": {
            "формула": "E = mc²",
            "змінні": {"E": "енергія спокою (Дж)", "m": "маса (кг)", "c": "швидкість світла (≈3×10⁸ м/с)"},
            "приклад": "Формула Ейнштейна, показує еквівалентність маси та енергії",
            "примітка": "Це формула енергії спокою, не враховує релятивістський приріст маси при русі",
        },
    },
    "хімія": {
        "густина речовини": {
            "формула": "ρ = m / V",
            "змінні": {"ρ": "густина (г/см³ або кг/м³)", "m": "маса (г)", "V": "об'єм (см³)"},
            "приклад": "Тіло масою 100 г і об'ємом 20 см³: ρ = 5 г/см³",
        },
        "молярна маса": {
            "формула": "M = m / n",
            "змінні": {"M": "молярна маса (г/моль)", "m": "маса (г)", "n": "кількість речовини (моль)"},
            "приклад": "10 г речовини — 0.5 моль: M = 20 г/моль",
        },
        "кількість речовини": {
            "формула": "n = m / M",
            "змінні": {"n": "кількість речовини (моль)", "m": "маса (г)", "M": "молярна маса (г/моль)"},
            "приклад": "36 г води (M=18 г/моль): n = 2 моль",
            "примітка": "Обернена форма формули молярної маси",
        },
        "концентрація": {
            "формула": "C = n / V",
            "змінні": {"C": "молярна концентрація (моль/л)", "n": "кількість речовини (моль)", "V": "об'єм розчину (л)"},
            "приклад": "0.2 моль речовини в 2 л розчину: C = 0.1 моль/л",
        },
        "масова частка речовини": {
            "формула": "ω = m(речовини) / m(розчину) × 100%",
            "змінні": {"ω": "масова частка (%)", "m(речовини)": "маса розчиненої речовини (г)", "m(розчину)": "загальна маса розчину (г)"},
            "приклад": "20 г солі в 200 г розчину: ω = 10%",
        },
        "закон авогадро": {
            "формула": "V = n × Vm",
            "змінні": {"V": "об'єм газу (л)", "n": "кількість речовини (моль)", "Vm": "молярний об'єм (≈22.4 л/моль за н.у.)"},
            "приклад": "2 моль газу за нормальних умов: V = 44.8 л",
            "примітка": "Діє лише за нормальних умов (0°C, 101.3 кПа)",
        },
        "рн розчину": {
            "формула": "pH = -log[H⁺]",
            "змінні": {"pH": "водневий показник", "[H⁺]": "концентрація іонів водню (моль/л)"},
            "приклад": "[H⁺] = 10⁻³ моль/л: pH = 3",
            "примітка": "pH < 7 — кисле середовище, pH = 7 — нейтральне, pH > 7 — лужне",
        },
        "ступінь дисоціації": {
            "формула": "α = Nдис / N × 100%",
            "змінні": {"α": "ступінь дисоціації (%)", "Nдис": "число дисоційованих молекул", "N": "загальне число молекул"},
            "приклад": "З 1000 молекул дисоціювало 50: α = 5%",
        },
        "вихід продукту реакції": {
            "формула": "η = mпракт / mтеор × 100%",
            "змінні": {"η": "вихід продукту (%)", "mпракт": "практично отримана маса (г)", "mтеор": "теоретично можлива маса (г)"},
            "приклад": "Теоретично 10 г, отримано 8.5 г: η = 85%",
        },
        "закон бойля-маріотта": {
            "формула": "p₁V₁ = p₂V₂",
            "змінні": {"p₁, V₁": "початковий тиск і об'єм", "p₂, V₂": "кінцевий тиск і об'єм"},
            "приклад": "При постійній температурі: якщо тиск зростає вдвічі, об'єм зменшується вдвічі",
            "примітка": "Діє при сталій температурі (ізотермічний процес)",
        },
    },
}

PERIODIC_TABLE = {
    "H":  {"name_ua":"Гідроген","name_en":"Hydrogen","number":1,"mass":1.008,"group":"неметал",
           "electron_config":"1s¹","properties":"Найлегший елемент, горючий газ без кольору та запаху","uses":"Виробництво аміаку, водневе паливо"},
    "He": {"name_ua":"Гелій","name_en":"Helium","number":2,"mass":4.003,"group":"інертний газ",
           "electron_config":"1s²","properties":"Інертний газ, другий за легкістю, не горить","uses":"Наповнення кульок, охолодження надпровідників"},
    "C":  {"name_ua":"Карбон","name_en":"Carbon","number":6,"mass":12.011,"group":"неметал",
           "electron_config":"[He] 2s² 2p²","properties":"Основа органічних сполук, графіт, алмаз","uses":"Сталь, органічна хімія, активоване вугілля"},
    "N":  {"name_ua":"Нітроген","name_en":"Nitrogen","number":7,"mass":14.007,"group":"неметал",
           "electron_config":"[He] 2s² 2p³","properties":"Складає 78% атмосфери","uses":"Аміак, добрива, рідкий азот"},
    "O":  {"name_ua":"Оксиген","name_en":"Oxygen","number":8,"mass":15.999,"group":"неметал",
           "electron_config":"[He] 2s² 2p⁴","properties":"Підтримує горіння і дихання","uses":"Дихання, медицина, металургія"},
    "F":  {"name_ua":"Флуор","name_en":"Fluorine","number":9,"mass":18.998,"group":"неметал",
           "electron_config":"[He] 2s² 2p⁵","properties":"Найактивніший неметал, отруйний газ","uses":"Зубні пасти, тефлон, холодоагенти"},
    "Ne": {"name_ua":"Неон","name_en":"Neon","number":10,"mass":20.180,"group":"інертний газ",
           "electron_config":"[He] 2s² 2p⁶","properties":"Інертний, світиться при розряді","uses":"Неонові лампи, реклама"},
    "Na": {"name_ua":"Натрій","name_en":"Sodium","number":11,"mass":22.990,"group":"лужний метал",
           "electron_config":"[Ne] 3s¹","properties":"М'який метал, бурхливо реагує з водою","uses":"Кухонна сіль (NaCl), натрієві лампи"},
    "Mg": {"name_ua":"Магній","name_en":"Magnesium","number":12,"mass":24.305,"group":"лужноземельний метал",
           "electron_config":"[Ne] 3s²","properties":"Легкий, горить яскравим білим полум'ям","uses":"Сплави, фотографія, ліки"},
    "Al": {"name_ua":"Алюміній","name_en":"Aluminium","number":13,"mass":26.982,"group":"метал",
           "electron_config":"[Ne] 3s² 3p¹","properties":"Легкий, стійкий до корозії","uses":"Авіабудування, упаковка, посуд"},
    "Si": {"name_ua":"Силіцій","name_en":"Silicon","number":14,"mass":28.085,"group":"неметал",
           "electron_config":"[Ne] 3s² 3p²","properties":"Напівпровідник, основа скла","uses":"Мікрочипи, скло, сонячні панелі"},
    "P":  {"name_ua":"Фосфор","name_en":"Phosphorus","number":15,"mass":30.974,"group":"неметал",
           "electron_config":"[Ne] 3s² 3p³","properties":"Існує у білій і червоній формах","uses":"Добрива, сірники, ДНК"},
    "S":  {"name_ua":"Сульфур","name_en":"Sulfur","number":16,"mass":32.06,"group":"неметал",
           "electron_config":"[Ne] 3s² 3p⁴","properties":"Жовтий, крихкий, характерний запах сполук","uses":"Сірчана кислота, гума, добрива"},
    "Cl": {"name_ua":"Хлор","name_en":"Chlorine","number":17,"mass":35.45,"group":"неметал",
           "electron_config":"[Ne] 3s² 3p⁵","properties":"Жовто-зелений отруйний газ","uses":"Дезінфекція води, ПВХ, відбілювачі"},
    "K":  {"name_ua":"Калій","name_en":"Potassium","number":19,"mass":39.098,"group":"лужний метал",
           "electron_config":"[Ar] 4s¹","properties":"М'який, дуже реакційноздатний","uses":"Добрива, скло, ліки"},
    "Ca": {"name_ua":"Кальцій","name_en":"Calcium","number":20,"mass":40.078,"group":"лужноземельний метал",
           "electron_config":"[Ar] 4s²","properties":"Основа кісток і зубів","uses":"Будівництво (цемент), харчова добавка"},
    "Fe": {"name_ua":"Ферум","name_en":"Iron","number":26,"mass":55.845,"group":"перехідний метал",
           "electron_config":"[Ar] 3d⁶ 4s²","properties":"Магнітний, легко іржавіє","uses":"Сталь, будівництво, гемоглобін крові"},
    "Cu": {"name_ua":"Купрум","name_en":"Copper","number":29,"mass":63.546,"group":"перехідний метал",
           "electron_config":"[Ar] 3d¹⁰ 4s¹","properties":"Гарний провідник тепла і струму","uses":"Проводка, сплави, посуд"},
    "Zn": {"name_ua":"Цинк","name_en":"Zinc","number":30,"mass":65.38,"group":"перехідний метал",
           "electron_config":"[Ar] 3d¹⁰ 4s²","properties":"Сріблясто-сірий, захищає залізо від корозії","uses":"Оцинкування, батарейки, сплави"},
    "Au": {"name_ua":"Золото","name_en":"Gold","number":79,"mass":196.967,"group":"перехідний метал",
           "electron_config":"[Xe] 4f¹⁴ 5d¹⁰ 6s¹","properties":"Жовтий, не окислюється, пластичний","uses":"Ювелірні вироби, електроніка"},
}
def convert_physics(value: float, from_unit: str, to_unit: str) -> float:
    """Виконує конвертацію фізичної величини за відомою парою одиниць."""
    key = (from_unit.strip(), to_unit.strip())
    conversions = {
        ("км/год", "м/с"):   value / 3.6,
        ("м/с", "км/год"):   value * 3.6,
        ("C", "F"):          value * 9 / 5 + 32,
        ("F", "C"):          (value - 32) * 5 / 9,
        ("C", "K"):          value + 273.15,
        ("K", "C"):          value - 273.15,
        ("Дж", "кал"):       value / 4.184,
        ("кал", "Дж"):       value * 4.184,
        ("атм", "Па"):       value * 101_325,
        ("Па", "атм"):       value / 101_325,
    }
    if key not in conversions:
        return None
    return conversions[key]
total_formulas = sum(len(f) for f in FORMULAS_DB.values())
print(f"✅ База формул: {total_formulas} формул ({', '.join(FORMULAS_DB.keys())})")
print(f"✅ Періодична таблиця: {len(PERIODIC_TABLE)} елементів")


✅ База формул: 30 формул (математика, фізика, хімія)
✅ Періодична таблиця: 20 елементів


### ✍️ Продуктовий коментар

Конвертер фізичних одиниць реалізовано через звичайний словник із готовими значеннями, а не через lambda-функції. Lambda — це "функція на льоту", ще не порахований результат, а шматок коду. Якщо LangChain або агент спробує серіалізувати (перетворити в текст/JSON для логів чи передачі) такий об'єкт, це може викликати помилку, бо lambda — це код, а не звичайні дані. Готове порахуване значення в словнику — це просто дані, вони серіалізуються без проблем у будь-якому місці системи.

FORMULAS_DB побудована як вкладений словник (предмет → назва формули → деталі), а не плоский список. Це дозволяє інструменту, який працюватиме з цими даними, легко фільтрувати за предметом і шукати за назвою, не розбираючи текст вручну.

Питання про поведінку бота, коли формули немає в базі, і про те, чому бот не має вигадувати формули, свідомо винесені в продуктовий коментар після Кроку 5: саме там реалізований `formula_lookup`, і про цю поведінку можна говорити на конкретному коді, а не абстрактно.

## 2. Інструменти (@tool)

In [5]:
@tool
def formula_lookup(query: str) -> str:
    """Шукає формули з математики, фізики, хімії.
    Використовуй коли учень питає формулу, як обчислити, пояснити формулу.
    Приклад: 'кінетична енергія', 'теорема піфагора'"""
    if not query.strip():
        return "Вкажи назву формули або тему."
    q = query.lower()
    for subject, formulas in FORMULAS_DB.items():
        for name, data in formulas.items():
            if q in name.lower() or name.lower() in q:
                result = [f"{name.upper()} ({subject})\n", f"Формула: {data['формула']}\n", "Змінні:"]
                for var, meaning in data.get("змінні", {}).items():
                    result.append(f"  {var} — {meaning}")
                if "приклад" in data: result.append(f"\nПриклад: {data['приклад']}")
                if "примітка" in data: result.append(f"\nПримітка: {data['примітка']}")
                return "\n".join(result)
    # часткові збіги
    partial = []
    for subject, formulas in FORMULAS_DB.items():
        for name in formulas:
            if any(w in name.lower() for w in q.split() if len(w) > 3):
                partial.append(f"  {name} ({subject})")
    if partial:
        return f"Точної формули не знайдено. Можливо:\n" + "\n".join(partial)
    return f"Формулу '{query}' не знайдено. Спробуй: математика, фізика, хімія."

@tool
def unit_converter_physics(query: str) -> str:
    """Конвертує фізичні одиниці: швидкість, температуру, енергію, тиск.
    Приклад: '100 км/год у м/с', '37 цельсій у фаренгейт'"""
    if not query.strip():
        return "Вкажи значення та одиниці. Наприклад: '100 км/год у м/с'."
    numbers = re.findall(r"-?\d+(?:[.,]\d+)?", query)
    if not numbers:
        return "Не знайдено числа."
    value = float(numbers[0].replace(",", "."))
    q = query.lower()
    unit_aliases = {
        "км/год": ["км/год","км/г","km/h"], "м/с": ["м/с","m/s"],
        "C": ["цельсій","°c","celsius"], "F": ["фаренгейт","°f","fahrenheit"],
        "K": ["кельвін"," k ","kelvin"], "Дж": ["джоуль","дж"],
        "кал": ["калорія","кал","cal"], "атм": ["атмосфера","атм"],
        "Па": ["паскаль","па","pascal"],
    }
    def detect(text):
        for canonical, aliases in unit_aliases.items():
            if any(a in text for a in aliases): return canonical
        return None
    from_unit = detect(q)
    parts = re.split(r"\s+[уводвдо]+\s+", q)
    to_unit = detect(parts[-1]) if len(parts) > 1 else None
    if not from_unit or not to_unit:
        return "Не розпізнано одиниці. Доступні: км/год↔м/с, C↔F↔K, Дж↔кал, атм↔Па"
    result = convert_physics(value, from_unit, to_unit)
    if result is None:
        return f"Конвертація {from_unit} → {to_unit} не підтримується."
    return f"{value} {from_unit} = {result:.4g} {to_unit}"

def _format_element(symbol, data):
    return (f"{data['name_ua']} ({data['name_en']}) — {symbol}\n\n"
            f"Атомний номер: {data['number']}\nМаса: {data['mass']} г/моль\n"
            f"Група: {data['group']}\nКонфігурація: {data['electron_config']}\n\n"
            f"Властивості: {data['properties']}\nЗастосування: {data['uses']}")

@tool
def periodic_table(query: str) -> str:
    """Інформація про хімічний елемент: маса, конфігурація, властивості.
    Приклад: 'Fe', 'залізо', 'золото'"""
    if not query.strip():
        return "Вкажи символ або назву елемента."
    q = query.lower().strip()
    for symbol, data in PERIODIC_TABLE.items():
        if q == symbol.lower(): return _format_element(symbol, data)
    for symbol, data in PERIODIC_TABLE.items():
        if q in data["name_ua"].lower() or data["name_ua"].lower() in q or q in data["name_en"].lower():
            return _format_element(symbol, data)
    available = ", ".join(f"{s} ({d['name_ua']})" for s, d in PERIODIC_TABLE.items())
    return f"Елемент '{query}' не знайдено. Доступні: {available}."

@tool
def study_planner(query: str) -> str:
    """Планує підготовку до іспиту: теми/день, чи вистачить часу.
    Приклад: '15 тем, 10 днів, 3 години на день'"""
    if not query.strip():
        return "Вкажи: кількість тем, днів, годин на день."
    numbers = re.findall(r"\d+(?:[.,]\d+)?", query)
    if len(numbers) < 3:
        return "Потрібно 3 числа: теми, дні, години. Приклад: '15 тем, 10 днів, 3 години'."
    topics = int(numbers[0]); days = int(numbers[1])
    hours = float(numbers[2].replace(",", "."))
    if topics <= 0 or days <= 0 or hours <= 0: return "Всі значення > 0."
    if hours > 16: return f"{hours} годин/день — нереалістично."
    q = query.lower()
    h_per_topic = 2.5 if "важк" in q else 1.0 if "легк" in q else 1.5
    difficulty = "важка" if "важк" in q else "легка" if "легк" in q else "середня"
    needed = topics * h_per_topic; available = days * hours
    result = [f"ПЛАН ПІДГОТОВКИ\n", f"Тем: {topics}, Днів: {days}, Годин/день: {hours}",
              f"Складність: {difficulty}\n",
              f"Потрібно: {needed:.0f} год | Доступно: {available:.0f} год"]
    if available >= needed:
        result.append(f"✅ Часу достатньо! Резерв: {available-needed:.0f} год")
    else:
        result.append(f"⚠️ Дефіцит: {needed-available:.0f} год")
    result += [f"\nТем/день: {topics/days:.1f}",
               f"Помодоро (25хв): {int(hours*60/25)} сесій/день",
               "\nПоради:", "1. Складне — зранку", "2. Активне повторення",
               "3. Останній день — повторення", "4. Сон ≥7 годин"]
    return "\n".join(result)

tools = [formula_lookup, unit_converter_physics, periodic_table, study_planner]
print("✅ Інструменти StudyMate створено.")


✅ Інструменти StudyMate створено.


### ✍️ Продуктовий коментар

`formula_lookup` спочатку шукає точний збіг, а якщо не знайдено — шукає часткові збіги за окремими словами запиту (довшими за 3 символи) і пропонує варіанти замість жорсткого "не знайдено". Це важливо для досвіду користувача: учень рідко формулює запит точно так, як названо у базі, і без часткового пошуку бот виглядав би значно "тупішим", ніж є насправді.

Якщо формули немає в базі взагалі (ні точного, ні часткового збігу), бот чесно відповідає, що не знайшов її, і пропонує уточнити запит або вказати предмет. Бот не намагається згадати чи придумати формулу з власних знань моделі. Для освітнього продукту це критично: якщо учень готується до контрольної і отримає впевнену, але вигадану формулу, він завчить неправильну інформацію, довіряючи боту. Чесна відмова гірша для сприйняття (менш "розумна" відповідь), але набагато безпечніша для реального результату навчання.

`study_planner` перевіряє реалістичність введених годин (`hours > 16`) — захист від абсурдного вводу, коли розрахунок формально спрацює, але видасть план, фізично неможливий для виконання. Важливо чесно зазначити: поріг 16 годин обраний умовно, як груба межа "точно нереально" (24 години на добу мінус сон і побут), а не як перевірена норма навчального навантаження. Для реального продукту цей поріг варто було б узгодити з методистом чи спиратись на дослідження продуктивності навчання, а не залишати довільним числом.

`_format_element` винесена окремою функцією без декоратора `@tool`, бо це не самостійний інструмент, а допоміжне форматування всередині `periodic_table`. Якщо додати їй `@tool`, агент міг би спробувати викликати її напряму як окремий інструмент, що не потрібно.

## 3. Модель

In [6]:
chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    max_tokens=512,
    timeout=30,
)

test = chat_model.invoke("Одним реченням підтвердь готовність працювати як освітній асистент.")
print("✅ Модель ОК:", test.content)

✅ Модель ОК: Я готовий працювати як освітній асистент і підтримувати навчальний процес.


### ✍️ Продуктовий коментар

Для освітнього бота обрано temperature=0.3, свідомо низьке значення. Висока варіативність тут ризикована: у розважальному чат-боті різні формулювання відповіді — це нормально і навіть цікаво, але для формули це неприпустимо, вона має звучати однаково точно при кожному повторному запиті, а не "творчо" переформульовуватись. Це узгоджується з попереднім експериментом (Tier 2, AI Fundamentals): температура впливає на стиль подачі, а не на правильність обчислень, тож підвищення температури не зробить відповідь "чіткішою", лише додасть варіативності формулювань.

max_tokens рекомендую збільшити до 700 (базове значення в шаблоні — 512), бо формули з описом змінних, прикладом і приміткою можуть бути довшими за прості відповіді. Тут є пряма вартісна розвилка: більше токенів на відповідь означає вищу вартість кожного виклику OpenAI API, і якщо продукт масштабувати на багатьох учнів, це прямо впливає на юніт-економіку.

Timeout окремо вартий уваги через реальний контекст використання: в Україні регулярні перебої електропостачання і зв'язку роблять довші затримки відповіді ймовірнішими, ніж у стабільних умовах. Тому має сенс тримати timeout з певним запасом, а не мінімальним, інакше учень отримає обрив запиту саме в момент нестабільного зв'язку, коли й так найважче отримати відповідь.Пропоную збільшити до 60
Спецсимволи. У базі формули записані юнікодом (Eₖ = mv² / 2, πr², λ = v / f), але в реальних відповідях модель не передавала цей рядок як є, а переписувала його у власну нотацію LaTeX: \[ E_k = \frac{mv^2}{2} \]. Це видно в усіх тестах з формулами. Наслідок конкретний: у середовищі, де LaTeX рендериться (Colab, Jupyter), учень бачить нормальну формулу, а в звичайному чаті без рендера він побачить набір слешів і дужок замість формули. Крім того, юнікодні індекси (ₖ, ₚ, ₙ) розбиваються на окремі токени, тобто витрачають більше токенів, ніж звичайні літери. Що з цим робити: або явно вказати в системному промпті формат виводу формул (тільки простий текст, без LaTeX), або одразу зберігати в базі два варіанти запису і рендерити потрібний залежно від того, де саме показується відповідь.

## 4. Агент

In [7]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """
Ти — освітній асистент StudyMate, помічник для учнів і студентів точних наук.

Твої завдання:
- знаходити формули з математики, фізики та хімії і пояснювати їх;
- конвертувати фізичні одиниці вимірювання;
- надавати інформацію про хімічні елементи;
- допомагати планувати підготовку до іспитів;
- відповідати українською мовою, чітко і зрозуміло.

Правила використання інструментів:
- Коли питають формулу або як щось обчислити — використовуй formula_lookup.
- Коли потрібна конвертація фізичних величин — використовуй unit_converter_physics.
- Коли питають про хімічний елемент — використовуй periodic_table.
- Коли потрібно скласти план підготовки до іспиту — використовуй study_planner.

Важливо:
- Пояснюй, а не просто відповідай. Після формули завжди поясни, що означають змінні.
- Якщо учень просить "пояснити ще раз" або "простіше" — перефразуй відповідь іншими словами.
- Не вигадуй формули самостійно — використовуй лише інструмент formula_lookup.
- Якщо формули немає в базі — чесно скажи про це і запропонуй уточнити запит.
- Не видавай себе за джерело абсолютної істини: якщо не впевнений — скажи про це.
- Якщо запит виходить за межі точних наук, ввічливо поясни свою спеціалізацію.
"""

agent = create_agent(
    model=chat_model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Агент StudyMate створений.")

# Тестовий виклик
test_messages = [
    {"role": "user", "content": "Яка формула кінетичної енергії?"}
]
result = agent.invoke({"messages": test_messages})
last = result["messages"][-1]
print("\\n🤖 StudyMate (тест):")
print(last.content if hasattr(last, "content") else last.get("content", ""))


✅ Агент StudyMate створений.
\n🤖 StudyMate (тест):
Формула для обчислення кінетичної енергії виглядає так:

\[ E_k = \frac{mv^2}{2} \]

Де:
- \( E_k \) — кінетична енергія, вимірюється в джоулях (Дж);
- \( m \) — маса тіла, вимірюється в кілограмах (кг);
- \( v \) — швидкість тіла, вимірюється в метрах за секунду (м/с).

Наприклад, якщо у нас є тіло масою 2 кг, яке рухається зі швидкістю 3 м/с, ми можемо обчислити його кінетичну енергію так:

\[ E_k = \frac{2 \times (3)^2}{2} = \frac{2 \times 9}{2} = 9 \text{ Дж} \]

Це означає, що кінетична енергія цього тіла становить 9 джоулів.


### ✍️ Продуктовий коментар

**Чому в SYSTEM_PROMPT є рядок "не видавай себе за джерело абсолютної істини"?** Модель уже знає більшість шкільних формул зі свого навчання, і ніщо на рівні архітектури фізично не забороняє їй згенерувати формулу самостійно, якщо `formula_lookup` не знайшов точного збігу (нетипове формулювання, відсутня тема). Без цього правила є ризик, що модель тихо підставить формулу з власної пам'яті замість чесного "не знайдено", і подасть це з тією ж впевненою інтонацією, що й перевірену відповідь з бази. Учень не може відрізнити, звідки взялась формула, якщо бот сам про це не сигналізує. Для освітнього контексту ризик вищий, ніж для розважального бота: учень довіряє, готується до реальної контрольної, і неточна інформація має конкретну ціну, помилку на іспиті, а не просто невдалий жарт.

**"Якщо учень просить пояснити простіше — перефразуй" — інструкція для моделі чи для агента? Хто відповідає технічно?** Це інструкція для моделі, не для агента. Технічно `system_prompt` передається в `create_agent(...)` і стає частиною контексту, який бачить LLM при кожному виклику. Сам агент (оркестрація LangGraph, цикл виклику інструментів, керування списком `messages`) не містить окремого коду виду "якщо користувач просить простіше — зроби X". Всю роботу з перефразуванням виконує модель: вона читає системний промпт, бачить правило, і генерує новий текст своїми словами. Відповідальність тут повністю на моделі, а отже негарантована, немає детермінованого коду, який би примусово це виконав, лише інструкція текстом.

**Порівняння з SYSTEM_PROMPT варіанту FitCoach.** StudyMate має більше правил  (про чесність і межу між довідником і власними знаннями моделі): "не вигадуй формули", "якщо формули немає — чесно скажи", "не видавай себе за джерело абсолютної істини" — три окремі пункти. FitCoach обмежується одним загальним застереженням "не давай медичних рекомендацій — ти не лікар". Причина в природі домену: формула об'єктивно перевірювана, вона або правильна, або ні, і ціна помилки конкретна (неправильна відповідь на контрольній). Фітнес-рекомендації розмитіші за своєю природою, рідко є єдино правильна відповідь, тому там достатньо однієї захисної фрази, що знімає відповідальність, а не детальних інструкцій на кожен випадок невизначеності. Водночас FitCoach явно прописує правило "запам'ятай параметри користувача в контексті", якого немає текстом у StudyMate, хоча архітектурно пам'ять в обох ботах працює однаково через список `messages`, StudyMate просто не проговорює це окремим правилом.

## 5. Інтерактивний чат

In [8]:
def run_studymate_session():
    print("\n" + "=" * 60)
    print("📚 StudyMate — освітній AI-асистент")
    print("=" * 60)
    print("exit — завершити, /reset — скинути контекст")
    print("-" * 60)
    messages = []
    while True:
        try:
            user_input = input("\n👤 Ви: ").strip()
            if not user_input: continue
            if user_input.lower() in ("exit","вихід","/exit","quit"):
                print("👋 До побачення!"); break
            if user_input.lower() in ("/reset","reset"):
                messages = []; print("🔄 Контекст скинуто."); continue
            messages.append({"role": "user", "content": user_input})
            print("🤔 Думаю...")
            result = agent.invoke({"messages": messages})
            messages = result["messages"]
            last = messages[-1]
            ans = last.content if hasattr(last, "content") else last.get("content", "")
            print(f"\n🤖 {ans}")
        except KeyboardInterrupt:
            print("\n👋 Перервано."); break
        except Exception as e:
            print(f"❌ Помилка: {e}")

run_studymate_session()



📚 StudyMate — освітній AI-асистент
exit — завершити, /reset — скинути контекст
------------------------------------------------------------

👤 Ви: формула кінетичної енергії,
🤔 Думаю...

🤖 Формула для обчислення кінетичної енергії виглядає так:

\[ E_k = \frac{mv^2}{2} \]

Де:
- \( E_k \) — кінетична енергія, вимірюється в джоулях (Дж);
- \( m \) — маса тіла, вимірюється в кілограмах (кг);
- \( v \) — швидкість тіла, вимірюється в метрах за секунду (м/с).

Наприклад, якщо у нас є тіло масою 2 кг, яке рухається зі швидкістю 3 м/с, ми можемо обчислити його кінетичну енергію так:

\[ E_k = \frac{2 \times (3)^2}{2} = \frac{2 \times 9}{2} = 9 \, \text{Дж} \]

Отже, кінетична енергія цього тіла становить 9 джоулів.

👤 Ви: поясни ще раз простіше
🤔 Думаю...

🤖 Звичайно! Формула для кінетичної енергії виглядає так:

\[ E_k = \frac{mv^2}{2} \]

Ось що означають ці символи:
- \( E_k \) — це кінетична енергія, яку ми вимірюємо в джоулях (Дж).
- \( m \) — це маса тіла в кілограмах (кг).
- \( v \) 

### ✍️ Продуктовий коментар

**Як технічно реалізована "пам'ять"?** Пам'ять реалізована не через якийсь спеціальний механізм запам'ятовування, а просто через список `messages`, який накопичує всю історію діалогу і цілком передається в `agent.invoke({"messages": messages})` при кожному новому виклику. Коли учень пише "поясни ще раз простіше", модель бачить у контексті попереднє повідомлення користувача і свою попередню відповідь про кінетичну енергію, і саме тому розуміє, що саме пояснювати, без цього список був би порожнім і бот перепитав би "що саме пояснити".

Реальний тест підтвердив це: на другий запит бот **не викликав `formula_lookup` повторно** (я перевірив, відповідь не була ідентичним форматованим виводом інструменту), а сам перегенерував пояснення, розбивши обчислення на покрокові пункти (1, 2, 3) замість суцільної формули. Втім, чесно кажучи, "простіше" тут вийшло радше "детальніше по кроках", ніж терміноогічно простіше, формула, LaTeX-запис і одиниці виміру (Дж, кг, м/с) лишились ті самі. Тобто перефразування спрацювало, але не так глибоко, як могла б очікувати людина, що не розуміє нотацію взагалі.

**Різниця /reset для StudyMate і FitCoach.** У FitCoach `/reset` стирає введені параметри (вагу, зріст), при наступному запиті на калорії користувачу доведеться ввести їх знову, незручно, але легко відновлювано (два числа). У StudyMate `/reset` стирає не просто параметри, а зв'язок між питаннями в ланцюжку міркування, наприклад, якщо учень кілька разів уточнював план підготовки, посилаючись на попередні теми й дні, після `/reset` цей ланцюжок зникає, і учню доведеться заново формулювати весь контекст задачі. Скидання шкідливіше саме для StudyMate у сценаріях багатокрокового планування, а не в разових питаннях про формулу.

**Зростання контексту при 25 повідомленнях.** Список `messages` передається в `agent.invoke()` цілком при кожному виклику, тобто 25-те повідомлення оплачує не лише себе, а й усі 24 попередніх як вхідні токени заново. Вартість і затримка відповіді ростуть кумулятивно, а не залишаються сталими на кожен окремий запит. Це той самий принцип, що я вже фіксував у попередньому завданні (HW3) про ризики зростання контексту, тепер видно його на реально працюючому коді.

## 6. Автоматичне тестування

In [10]:
def run_automatic_tests():
    """
    Автоматичне тестування типових сценаріїв StudyMate.
    Структура аналогічна run_automatic_tests() з практичної роботи (крок 8).
    """
    test_queries = [
        # 1. Звичайне запитання без tool
        "Привіт! Що ти вмієш робити?",
        # 2. Пошук формули (formula_lookup)
        "Яка формула кінетичної енергії?",
        # 3. Продовження контексту — прохання пояснити простіше
        "Поясни цю формулу простішими словами.",
        # 4. Конвертація одиниць (unit_converter_physics)
        "Переведи 100 км/год у м/с.",
        # 5. Інформація про елемент (periodic_table)
        "Розкажи про елемент залізо.",
        # 6. Планування підготовки (study_planner)
        "Допоможи спланувати підготовку до іспиту: 15 тем, 10 днів, 3 години на день.",
        # 7. Перевірка пам'яті — посилання на попередній контекст
        "А чи вистачить часу, якщо у мене лише 7 днів?",
        # 8. Формула, якої немає в базі — перевірка чесності бота
        "Яка формула квантової заплутаності?",
        # 9. Запит поза темою — перевірка меж поведінки
        "Порадь мені фільм на вечір.",
        # 10. Некоректний або неповний запит
        "Переведи 100 у м/с.",
    ]

    messages = []

    print("\\n" + "=" * 60)
    print("🧪 АВТОМАТИЧНЕ ТЕСТУВАННЯ StudyMate")
    print("=" * 60)

    for i, query in enumerate(test_queries, start=1):
        print(f"\\nТест #{i}")
        print("-" * 60)
        print(f"👤 Запит: {query}")

        messages.append({"role": "user", "content": query})

        try:
            result_state = agent.invoke({"messages": messages})
            messages = result_state["messages"]
            last_message = messages[-1]
            answer = (
                last_message.content
                if hasattr(last_message, "content")
                else last_message.get("content", "")
            )

            print("✅ Відповідь StudyMate:")
            print(answer)

        except Exception as e:
            print(f"❌ Помилка: {e}")

        print("-" * 60)

    print("\\n✅ Тестування завершено.")

run_automatic_tests()


\n============================================================
🧪 АВТОМАТИЧНЕ ТЕСТУВАННЯ StudyMate
\nТест #1
------------------------------------------------------------
👤 Запит: Привіт! Що ти вмієш робити?
✅ Відповідь StudyMate:
Привіт! Я можу допомогти з різними аспектами точних наук, такими як:

- Знаходити формули з математики, фізики та хімії і пояснювати їх.
- Конвертувати фізичні одиниці вимірювання.
- Надавати інформацію про хімічні елементи.
- Допомагати планувати підготовку до іспитів.

Якщо у тебе є конкретне питання або завдання, не соромся запитувати!
------------------------------------------------------------
\nТест #2
------------------------------------------------------------
👤 Запит: Яка формула кінетичної енергії?
✅ Відповідь StudyMate:
Формула для обчислення кінетичної енергії виглядає так:

\[ E_k = \frac{mv^2}{2} \]

Де:
- \( E_k \) — кінетична енергія, вимірюється в джоулях (Дж);
- \( m \) — маса об'єкта, вимірюється в кілограмах (кг);
- \( v \) — швидкість об'єк

### ✍️ Продуктовий коментар

**Тест №3 — "поясни простіше" без уточнення.** Бот визначає, що саме пояснювати, виключно через список `messages`: у контексті лишається попереднє повідомлення користувача і власна попередня відповідь про кінетичну енергію. Це спрацьовує надійно, доки ланцюжок коротки і однозначний. Але це не гарантія: якщо між формулою і проханням "поясни простіше" вклинилося б інше питання (наприклад, про елемент), не зрозуміло, чи бот однозначно визначить, яку саме з попередніх відповідей перефразовувати.

**Тест №7 — зміна кількості днів без повторення інших параметрів.** Бот успішно підхопив контекст: використав 15 тем і 3 години на день з попереднього повідомлення, підставивши лише нове число днів (7 замість 10), і коректно перерахував дефіцит часу. Це працює завдяки тому, що весь список `messages` передається агенту цілком при кожному виклику.

**Тест №8 і тест №10 — межові сценарії, найважливіша знахідка.** Тест №8 (формула, якої немає в базі) бот формально відхилив коректно ("не знайшов формулу"), але одразу ж пояснив суть квантової заплутаності з власних знань моделі, в обхід `formula_lookup` і всупереч правилу "не вигадуй". Тест №10 виявився ще показовішим: запит "Переведи 100 у м/с" без вказаної вихідної одиниці мав викликати повідомлення "не розпізнано одиниці" (це прописано в коді `unit_converter_physics`), але бот замість цього дослівно повторив відповідь з тесту №4 (100 км/год у м/с), фактично домисливши пропущену одиницю з попереднього контексту, а не з поточного запиту. Обидва випадки, різними шляхами, показують один системний ризик: інструкції в системному промпті ("не вигадуй", "якщо не впевнений, скажи про це") є текстовою рекомендацією для моделі, а не технічною гарантією. Агент має свободу вирішувати, викликати інструмент заново чи відповісти з пам'яті, і в цих двох випадках він обрав відповісти впевнено, хоча дані для впевненості були неповні. Це емпіричний доказ архітектурного слабкого місця, яке я вже фіксував раніше на етапі продуктового брифу (HW2): межа між довідником і власними знаннями моделі розмита, і жоден рядок промпту цього не гарантує технічно.

### Підсумкова продуктова таблиця

| Поле | Заповнення |
|---|---|
| Назва бота | StudyMate |
| Для кого він створений | Учні старшої школи і студенти перших курсів,<br>яким потрібно швидко знайти формулу з математики<br>чи фізики, перевести фізичну величину або<br>спланувати підготовку до іспиту з урахуванням<br>дедлайну. Хімічні інструменти (`periodic_table`,<br>формули з хімії) реалізовані через технічну<br>вимогу варіанту 3, а не як цільова цінність<br>продукту, див. примітку в Кроці 4 |
| Яку задачу вирішує | Замінює розрізнений пошук по конспектах,<br>підручниках і форумах на один діалоговий<br>інтерфейс, який дає перевірену відповідь<br>(формула, конвертація) або структурований<br>розрахунок (план підготовки), а не просто<br>посилання на джерело |
| Які tools реалізовано | `formula_lookup` (30 формул: математика,<br>фізика, хімія)<br>`unit_converter_physics` (швидкість,<br>температура, енергія, тиск)<br>`periodic_table` (20 елементів)<br>`study_planner` (план підготовки з<br>перевіркою реалістичності вводу) |
| Основна цінність для користувача | Бот не просто видає формулу, а пояснює її<br>(змінні, приклад), і зберігає контекст<br>діалогу, тож можна уточнювати ("поясни<br>простіше", "а якщо 7 днів") без повторення<br>всіх вхідних даних заново |
| Головний ризик або обмеження | Емпірично підтверджено тестуванням (тест №8<br>і тест №10): межа між пошуком у перевіреній<br>базі і власними знаннями моделі розмита.<br>Якщо формули немає в базі, або дані запиту<br>неповні, модель схильна впевнено підставити<br>відповідь із власної пам'яті чи домислити<br>пропущений параметр з попереднього<br>контексту. Інструкція текстом не є<br>технічною гарантією поведінки |
| Що варто покращити перед реальним використанням | Замінити текстову інструкцію "не вигадуй"<br>на технічний контроль: перевіряти, чи<br>відповідь агента фактично спирається на<br>виклик tool, а не на пряму генерацію, і<br>показувати учню позначку джерела ("з<br>довідника" / "пояснення моделі, не<br>перевірено"), особливо для запитів на межі<br>бази даних, як у тесті №10 |

### Таблиця тестування

| № | Тип сценарію | Запит | Що перевіряється | Результат |
|---|---|---|---|---|
| 1 | Звичайне запитання<br>без tool | Привіт! Що ти<br>вмієш робити? | Чи коректно бот<br>описує можливості<br>без виклику tools | Перелічив 4 функції,<br>tool не викликано.<br>Коректно |
| 2 | Виклик<br>formula_lookup | Яка формула<br>кінетичної<br>енергії? | Чи спрацював<br>пошук формули з<br>поясненням змінних | Формула, змінні,<br>приклад видані<br>правильно. Коректно |
| 3 | Продовження<br>контексту | Поясни цю<br>формулу<br>простішими<br>словами. | Чи бот пам'ятає,<br>що пояснювати, без<br>повторного tool | Перефразував без<br>повторного виклику.<br>Радше детальніше,<br>ніж простіше |
| 4 | Виклик<br>unit_converter_physics | Переведи 100<br>км/год у м/с. | Чи спрацювала<br>конвертація одиниць | 100 км/год = 27.78<br>м/с. Правильно |
| 5 | Виклик<br>periodic_table | Розкажи про<br>елемент<br>залізо. | Чи спрацював<br>пошук за<br>українською назвою | Номер, маса,<br>конфігурація,<br>властивості видані.<br>Коректно |
| 6 | Виклик<br>study_planner | Допоможи<br>спланувати:<br>15 тем, 10<br>днів, 3 год/день. | Чи правильно<br>розрахований план<br>і резерв часу | Потрібно 22 год,<br>доступно 30 год,<br>резерв 8 год.<br>Правильно |
| 7 | Продовження<br>контексту | А чи вистачить<br>часу, якщо у<br>мене лише 7<br>днів? | Чи бот підхопив<br>теми і години з<br>попереднього кроку | Підставив 7 днів,<br>зберіг 15 тем і 3<br>год/день. Пам'ять<br>спрацювала правильно |
| 8 | Межа<br>можливостей | Яка формула<br>квантової<br>заплутаності? | Чи бот чесно<br>визнає відсутність<br>формули в базі | Сказав "не знайшов",<br>але пояснив суть з<br>власних знань.<br>Часткове порушення |
| 9 | Запит поза<br>темою | Порадь мені<br>фільм на<br>вечір. | Чи бот тримається<br>меж спеціалізації | Ввічливо відмовив,<br>нагадав про<br>спеціалізацію.<br>Коректно |
| 10 | Некоректний або<br>неповний запит | Переведи 100<br>у м/с. | Чи бот попросить<br>уточнити відсутню<br>вихідну одиницю | Не уточнив, повторив<br>відповідь тесту №4,<br>домисливши одиницю.<br>Найважливіша знахідка |